# 02. Evidence-selection uncertainty

Repeatedly select evidence sentences and quantify selection instability.

In [1]:
from pathlib import Path
from itertools import combinations
import json
import re
import numpy as np
import pandas as pd
import requests
from openai import OpenAI

## 1. Configuration

In [2]:
def find_project_root():
    cwd = Path.cwd().resolve()
    for path in [cwd] + list(cwd.parents):
        if (path / "outputs").exists():
            return path
    return cwd

PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PROCESSED_DIR = OUTPUT_DIR / "processed_data"

# True for a sample run of 3 cases else full run.
SAMPLE_MODE = False
SAMPLE_CASES = 3

RUN_ARCH_TEST = False
RUN_BIOASQ_TEST = False

RUN_NAME = "sample" if SAMPLE_MODE else "full"
EVIDENCE_DIR = OUTPUT_DIR / "evidence_selection" / RUN_NAME
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

BACKEND = "ollama" if SAMPLE_MODE else "vllm"

OLLAMA_MODEL = "gemma3:12b"
VLLM_MODEL = "google/gemma-3-12b-it"

MODEL = (
    OLLAMA_MODEL
    if BACKEND == "ollama"
    else VLLM_MODEL
)

VLLM_BASE_URL = "http://localhost:8000/v1"

N_RUNS = 10
TEMPERATURE = 0.7
TOP_P = 0.9
NUM_CTX = 32768

BASE_SEED = 1000
RUN_SEEDS = [BASE_SEED + i for i in range(N_RUNS)]

REPRESENTATIVE_METHOD = "majority"

RAW_PATH = EVIDENCE_DIR / "evidence_runs.jsonl"
SUMMARY_PATH = EVIDENCE_DIR / "evidence_summary.csv"

## 2. Shared functions

In [3]:
def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def append_jsonl(record, path):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def format_sentences(sentences):
    return "\n".join(
        f"[{s['sentence_id']}] {s['text']}"
        for s in sentences
    )

def build_prompt(case):
    sentence_block = format_sentences(case["sentences"])

    if case["dataset"] == "archehr_qa":
        question_block = f"""
Patient question:
{case["patient_question"]}

Clinician-interpreted question:
{case["clinician_question"]}

Clinical specialty:
{case["clinical_specialty"]}
""".strip()
    else:
        question_block = f"""
Question:
{case["question"]}
""".strip()

    return f"""
You are an expert clinical NLP assistant. Select the sentence(s) that contain the evidence needed to answer the question.

{question_block}

Candidate evidence sentences:
{sentence_block}

Instructions:
- Select sentence IDs that provide sufficient evidence to answer the question.
- Include sentences needed to support important details such as dates, values, medications, doses, or negations.
- Include only directly relevant and answer-bearing sentences.
- Do not include unrelated, generic, administrative, or only tangentially related sentences.
- If none of the sentences provide evidence needed to answer the question, return [].

Return only a JSON array of sentence ID numbers, for example: [1, 5, 6]
""".strip()

def call_ollama(prompt, seed):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": TEMPERATURE,
                "top_p": TOP_P,
                "seed": seed,
                "num_predict": -1,
                "num_ctx": NUM_CTX,
            },
        },
        timeout=600,
    )
    response.raise_for_status()

    return " ".join(
        response.json()["response"].strip().split()
    )


def call_vllm(prompt, seed):
    response = requests.post(
        f"{VLLM_BASE_URL}/chat/completions",
        json={
            "model": MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "seed": seed,
            "stream": False,
        },
        timeout=600,
    )
    response.raise_for_status()

    text = (
        response.json()["choices"][0]["message"]["content"]
    )

    return " ".join(text.strip().split())


def call_model(prompt, seed):
    if BACKEND == "ollama":
        return call_ollama(
            prompt,
            seed=seed,
        )

    if BACKEND == "vllm":
        return call_vllm(
            prompt,
            seed=seed,
        )

    raise ValueError(f"Unknown backend: {BACKEND}")


def check_model_server():
    if BACKEND == "ollama":
        url = "http://localhost:11434/"
    else:
        url = f"{VLLM_BASE_URL}/models"

    response = requests.get(
        url,
        timeout=10,
    )
    response.raise_for_status()

def parse_selected_ids(text, valid_ids):
    text = text.strip()

    text = re.sub(
        r"^`{1,3}json\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"`{1,3}$",
        "",
        text,
    ).strip()

    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return [], False

    if not isinstance(data, list):
        return [], False

    valid_ids = set(map(str, valid_ids))
    selected = []

    for value in data:
        sid = str(value)

        if sid in valid_ids and sid not in selected:
            selected.append(sid)

    return selected, True

def jaccard_distance(a, b):
    a = set(a)
    b = set(b)

    if not a and not b:
        return 0.0

    return 1.0 - len(a & b) / len(a | b)

def average_pairwise_jaccard(runs):
    distances = [
        jaccard_distance(a, b)
        for a, b in combinations(runs, 2)
    ]
    return sum(distances) / len(distances) if distances else 0.0

def majority_vote(runs):
    counts = {}
    for run in runs:
        for sid in set(run):
            counts[sid] = counts.get(sid, 0) + 1

    selected = [
        sid
        for sid, count in counts.items()
        if count / len(runs) >= 0.5
    ]
    return sorted(selected, key=int)

def mean_distance_to_runs(selected, runs):
    return (
        sum(jaccard_distance(selected, run) for run in runs) / len(runs)
        if runs else 0.0
    )

def jaccard_medoid(runs):
    if not runs:
        return []

    distances = [
        mean_distance_to_runs(run, runs)
        for run in runs
    ]
    best_index = min(range(len(runs)), key=lambda i: (distances[i], i))
    return sorted(set(runs[best_index]), key=int)

def evidence_scores(predicted, gold):
    predicted = set(predicted)
    gold = set(gold)

    overlap = len(predicted & gold)
    precision = overlap / len(predicted) if predicted else 0.0
    recall = overlap / len(gold) if gold else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )
    return precision, recall, f1

In [4]:
VLLM_BASE_URL = "http://localhost:8000/v1"
MODEL = "google/gemma-3-12b-it"

client = OpenAI(
    base_url=VLLM_BASE_URL,
    api_key="EMPTY",
)


def call_model(prompt, seed):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=TEMPERATURE,
        top_p=TOP_P,
        seed=seed,
    )

    return response.choices[0].message.content.strip()

## 3. Load cases

In [5]:
analysis_sets = {
    "bioasq_train": load_jsonl(
        PROCESSED_DIR / "bioasq_train_cases.jsonl"
    ),
    "archehr_train": load_jsonl(
        PROCESSED_DIR / "archehr_train_cases.jsonl"
    ),
}

if RUN_BIOASQ_TEST:
    analysis_sets["bioasq_test"] = load_jsonl(
        PROCESSED_DIR / "bioasq_test_cases.jsonl"
    )

if RUN_ARCH_TEST:
    analysis_sets["archehr_test"] = load_jsonl(
        PROCESSED_DIR / "archehr_test_cases.jsonl"
    )

if SAMPLE_MODE:
    analysis_sets = {
        name: cases[:SAMPLE_CASES]
        for name, cases in analysis_sets.items()
        if name in {"bioasq_train", "archehr_train"}
    }

for name, cases in analysis_sets.items():
    print(name, len(cases))

bioasq_train 973
archehr_train 20


## 4. Run repeated evidence selection

In [6]:
existing = load_jsonl(RAW_PATH)
done = {
    (row["analysis_set"], row["case_id"], int(row["run_id"]))
    for row in existing
}

try:
    check_model_server()
except Exception as exc:
    raise RuntimeError(
        f"Start {BACKEND} before running repeated evidence selection."
    ) from exc

for analysis_set, cases in analysis_sets.items():
    for case in cases:
        valid_ids = {
            str(s["sentence_id"])
            for s in case["sentences"]
        }

        for run_id in range(N_RUNS):
            key = (analysis_set, case["case_id"], run_id)
            if key in done:
                continue

            seed = RUN_SEEDS[run_id]

            raw_response = call_model(
                build_prompt(case),
                seed=seed,
            )


            selected_ids, parse_ok = parse_selected_ids(
                raw_response,
                valid_ids,
            )

            record = {
                "analysis_set": analysis_set,
                "case_id": case["case_id"],
                "run_id": run_id,
                "seed": seed,
                "selected_sentence_ids": selected_ids,
                "parse_ok": parse_ok,
                "raw_response": raw_response,
            }

            append_jsonl(record, RAW_PATH)
            done.add(key)

            print(
                analysis_set,
                case["case_id"],
                run_id + 1,
                selected_ids,
            )

RuntimeError: Start vllm before running repeated evidence selection.

## 5. Calculate evidence-selection uncertainty

In [7]:
if REPRESENTATIVE_METHOD not in {"majority", "medoid"}:
    raise ValueError("REPRESENTATIVE_METHOD must be 'majority' or 'medoid'.")

runs_df = pd.DataFrame(load_jsonl(RAW_PATH))
summary_rows = []

for analysis_set, cases in analysis_sets.items():
    for case in cases:
        case_runs = runs_df[
            (runs_df["analysis_set"] == analysis_set)
            & (runs_df["case_id"] == case["case_id"])
        ].sort_values("run_id")

        if len(case_runs) != N_RUNS:
            raise ValueError(
                f"{case['case_id']}: expected {N_RUNS} runs, "
                f"found {len(case_runs)}"
            )

        if not case_runs["parse_ok"].all():
            raise ValueError(
                f"{case['case_id']}: one or more evidence outputs failed to parse."
            )

        runs = case_runs["selected_sentence_ids"].tolist()

        majority_ids = majority_vote(runs)
        medoid_ids = jaccard_medoid(runs)

        representative_ids = (
            majority_ids
            if REPRESENTATIVE_METHOD == "majority"
            else medoid_ids
        )

        if case["gold_evidence_complete"]:
            majority_p, majority_r, majority_f1 = evidence_scores(
                majority_ids,
                case["gold_evidence_ids"],
            )
            medoid_p, medoid_r, medoid_f1 = evidence_scores(
                medoid_ids,
                case["gold_evidence_ids"],
            )
            precision, recall, f1 = evidence_scores(
                representative_ids,
                case["gold_evidence_ids"],
            )
        else:
            majority_p = majority_r = majority_f1 = np.nan
            medoid_p = medoid_r = medoid_f1 = np.nan
            precision = recall = f1 = np.nan

        summary_rows.append({
            "analysis_set": analysis_set,
            "case_id": case["case_id"],
            "evidence_uncertainty": average_pairwise_jaccard(runs),
            "selected_sentence_ids": json.dumps(representative_ids),
            "evidence_precision": precision,
            "evidence_recall": recall,
            "evidence_f1": f1,
            "majority_sentence_ids": json.dumps(majority_ids),
            "medoid_sentence_ids": json.dumps(medoid_ids),
            "majority_mean_distance": mean_distance_to_runs(
                majority_ids, runs
            ),
            "medoid_mean_distance": mean_distance_to_runs(
                medoid_ids, runs
            ),
            "majority_f1": majority_f1,
            "medoid_f1": medoid_f1,
        })

evidence_summary = pd.DataFrame(summary_rows)
evidence_summary.to_csv(SUMMARY_PATH, index=False)

print("Saved:", SUMMARY_PATH)
display(
    evidence_summary.groupby("analysis_set")[
        [
            "evidence_uncertainty",
            "evidence_recall",
            "evidence_f1",
            "majority_f1",
            "medoid_f1",
        ]
    ].mean()
)

Saved: /Users/sangbin/Desktop/Dissertation/Msc_Dissertation/outputs/evidence_selection/full/evidence_summary.csv


,evidence_uncertainty,evidence_recall,evidence_f1,majority_f1,medoid_f1
analysis_set,,,,,
archehr_train,0.065459,0.497846,0.519643,0.519643,0.519643
bioasq_train,0.144070,0.562577,0.534488,0.534488,0.533163
